In [ ]:
# 자동 재주문 발주 프로그램 interface

import argparse
import sqlite3
from collections import defaultdict
from datetime import datetime
from pathlib import Path


# 프로그램이 자동으로 찾아볼 DB 파일 이름
DEFAULT_DB_NAMES = ["파리바게트.db", "paris_baguette.db", "database.db"]


class AutoReorderProgram:
    def __init__(self, db_path):
        self.db_path = Path(db_path)
        self.conn = None
        self.created_orders = []
        self.skipped_items = []
        self.plan_groups = {}

    # 1. DB 연결
    def connect(self):
        if not self.db_path.exists():
            raise FileNotFoundError(f"DB 파일을 찾을 수 없습니다: {self.db_path}")

        self.conn = sqlite3.connect(self.db_path)
        self.conn.row_factory = sqlite3.Row
        self.conn.execute("PRAGMA foreign_keys = ON;")

        self.validate_required_tables()
        print(f"\nDB 연결 완료: {self.db_path}")

    # 2. DB 연결 종료
    def close(self):
        if self.conn:
            self.conn.close()
            print("DB 연결을 종료했습니다.")

    # 3. 자동 재주문에 필요한 테이블이 있는지 확인
    def validate_required_tables(self):
        required = {"보유", "상품", "브랜드", "공급", "공급업체", "발주", "발주상세"}

        rows = self.fetch_all(
            """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
              AND name NOT LIKE 'sqlite_%';
            """
        )

        existing = {row["name"] for row in rows}
        missing = sorted(required - existing)

        if missing:
            raise RuntimeError("필요한 테이블이 없습니다: " + ", ".join(missing))

    # 4. SELECT 결과 여러 행 가져오기
    def fetch_all(self, sql, params=()):
        if self.conn is None:
            raise RuntimeError("DB 연결이 없습니다.")

        cur = self.conn.execute(sql, params)
        return cur.fetchall()

    # 5. SELECT 결과 한 행만 가져오기
    def fetch_one(self, sql, params=()):
        if self.conn is None:
            raise RuntimeError("DB 연결이 없습니다.")

        cur = self.conn.execute(sql, params)
        return cur.fetchone()

    # 6. 재고가 부족한 상품 찾기
    def scan_low_stock_items(self):
        rows = self.fetch_all(
            """
            SELECT
                h.지점명,
                h.상품코드,
                p.이름 AS 상품명,
                b.브랜드명,
                h.재고수량,
                h.최소재고기준,
                h.매장가격,
                p.가격 AS 기본가격,
                MIN(s.업체코드) AS 업체코드,
                sp.업체명
            FROM 보유 h
            JOIN 상품 p ON h.상품코드 = p.상품코드
            LEFT JOIN 브랜드 b ON p.브랜드코드 = b.브랜드코드
            LEFT JOIN 공급 s ON h.상품코드 = s.상품코드
            LEFT JOIN 공급업체 sp ON s.업체코드 = sp.업체코드
            WHERE h.재고수량 < h.최소재고기준
            GROUP BY
                h.지점명,
                h.상품코드,
                p.이름,
                b.브랜드명,
                h.재고수량,
                h.최소재고기준,
                h.매장가격,
                p.가격
            ORDER BY h.지점명, h.상품코드;
            """
        )

        candidates = []
        skipped = []

        # 7. 발주 가능한 상품과 제외할 상품 나누기
        for row in rows:
            store_name = row["지점명"]
            product_code = row["상품코드"]
            supplier_code = row["업체코드"]

            # 공급업체가 없으면 발주를 만들 수 없어서 제외
            if supplier_code is None:
                skipped.append(
                    {
                        "지점명": store_name,
                        "상품코드": product_code,
                        "상품명": row["상품명"],
                        "현재재고": row["재고수량"],
                        "최소재고기준": row["최소재고기준"],
                        "제외사유": "공급 가능한 업체가 없음",
                    }
                )
                continue

            # 이미 대기 중인 발주가 있으면 중복 발주를 막기 위해 제외
            if self.has_pending_order(store_name, product_code):
                skipped.append(
                    {
                        "지점명": store_name,
                        "상품코드": product_code,
                        "상품명": row["상품명"],
                        "현재재고": row["재고수량"],
                        "최소재고기준": row["최소재고기준"],
                        "제외사유": "이미 대기 중인 발주가 있음",
                    }
                )
                continue

            min_stock = row["최소재고기준"]
            current_stock = row["재고수량"]
            order_qty = self.calculate_order_quantity(current_stock, min_stock)

            candidates.append(
                {
                    "지점명": store_name,
                    "상품코드": product_code,
                    "상품명": row["상품명"],
                    "브랜드명": row["브랜드명"],
                    "현재재고": current_stock,
                    "최소재고기준": min_stock,
                    "주문수량": order_qty,
                    "업체코드": supplier_code,
                    "업체명": row["업체명"],
                }
            )

        self.skipped_items = skipped
        return candidates

    # 8. 같은 매장/상품에 대해 대기 중인 발주가 있는지 확인
    def has_pending_order(self, store_name, product_code):
        row = self.fetch_one(
            """
            SELECT 1
            FROM 발주 o
            JOIN 발주상세 od ON o.발주번호 = od.발주번호
            WHERE o.지점명 = ?
              AND od.상품코드 = ?
              AND o.상태 = '대기'
            LIMIT 1;
            """,
            (store_name, product_code),
        )

        return row is not None

    # 9. 주문할 수량 계산
    @staticmethod
    def calculate_order_quantity(current_stock, min_stock):
        target_stock = min_stock * 2
        qty = target_stock - current_stock
        return max(qty, min_stock - current_stock, 1)

    # 10. 같은 매장과 같은 공급업체끼리 묶기
    def build_order_plan(self, candidates):
        groups = defaultdict(list)

        for item in candidates:
            key = (item["지점명"], item["업체코드"], item["업체명"])
            groups[key].append(item)

        self.plan_groups = dict(groups)
        return self.plan_groups

    # 11. 미리보기 결과 출력
    def print_summary(self, candidates):
        print("\n" + "=" * 80)
        print("재고 확인 결과")
        print("=" * 80)

        total_low_stock = len(candidates) + len(self.skipped_items)

        print(f"재고가 부족한 상품 수: {total_low_stock}")
        print(f"발주를 만들 수 있는 상품 수: {len(candidates)}")
        print(f"이번에 제외된 상품 수: {len(self.skipped_items)}")

        if candidates:
            print("\n[발주 가능 상품]")
            print_dict_rows(candidates)

        if self.skipped_items:
            print("\n[제외된 상품]")
            print_dict_rows(self.skipped_items)

        groups = self.build_order_plan(candidates)

        print("\n[발주 예정 목록]")
        if not groups:
            print("새로 만들 발주가 없습니다.")
            return

        plan_rows = []
        for (store_name, supplier_code, supplier_name), items in groups.items():
            plan_rows.append(
                {
                    "지점명": store_name,
                    "업체코드": supplier_code,
                    "업체명": supplier_name,
                    "발주항목수": len(items),
                    "총주문수량": sum(item["주문수량"] for item in items),
                }
            )

        print_dict_rows(plan_rows)

    # 12. 실제로 발주와 발주상세 INSERT
    def execute_order_plan(self):
        if not self.plan_groups:
            print("새로 만들 발주가 없습니다.")
            return []

        now = datetime.now()
        order_date = now.strftime("%Y-%m-%d")
        created_orders = []

        try:
            self.conn.execute("BEGIN IMMEDIATE;")

            for (store_name, supplier_code, supplier_name), items in self.plan_groups.items():
                order_no = self.generate_order_no(len(created_orders) + 1)

                # 발주 1건 생성
                self.conn.execute(
                    """
                    INSERT INTO 발주 (발주번호, 발주일자, 상태, 지점명, 업체코드)
                    VALUES (?, ?, '대기', ?, ?);
                    """,
                    (order_no, order_date, store_name, supplier_code),
                )

                # 발주에 들어가는 상품 목록 생성
                for idx, item in enumerate(items, start=1):
                    self.conn.execute(
                        """
                        INSERT INTO 발주상세 (발주번호, 항목번호, 주문수량, 상품코드)
                        VALUES (?, ?, ?, ?);
                        """,
                        (order_no, idx, item["주문수량"], item["상품코드"]),
                    )

                created_orders.append(
                    {
                        "발주번호": order_no,
                        "발주일자": order_date,
                        "상태": "대기",
                        "지점명": store_name,
                        "업체코드": supplier_code,
                        "업체명": supplier_name,
                        "발주항목수": len(items),
                        "총주문수량": sum(item["주문수량"] for item in items),
                    }
                )

            self.conn.commit()

        except Exception:
            self.conn.rollback()
            raise

        self.created_orders = created_orders
        return created_orders

    # 13. 겹치지 않는 발주번호 만들기
    def generate_order_no(self, sequence):
        while True:
            stamp = datetime.now().strftime("%y%m%d%H%M%S")
            order_no = f"RO{stamp}{sequence:03d}"

            exists = self.fetch_one(
                """
                SELECT 발주번호
                FROM 발주
                WHERE 발주번호 = ?;
                """,
                (order_no,),
            )

            if not exists:
                return order_no

    # 14. 생성된 발주 출력
    def print_created_orders(self, created_orders):
        print("\n" + "=" * 80)
        print("생성된 발주")
        print("=" * 80)

        if not created_orders:
            print("생성된 발주가 없습니다.")
            return

        print_dict_rows(created_orders)

    # 15. 실행 결과 로그 저장
    def write_log(self, mode, candidates, created_orders):
        log_dir = Path.cwd() / "logs"
        log_dir.mkdir(exist_ok=True)
        log_file = log_dir / "auto_reorder_log.txt"

        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        lines = []
        lines.append("=" * 80)
        lines.append(f"실행시각: {now}")
        lines.append(f"실행모드: {mode}")
        lines.append(f"DB파일: {self.db_path}")
        lines.append(f"재고가 부족한 상품 수: {len(candidates) + len(self.skipped_items)}")
        lines.append(f"발주 가능한 상품 수: {len(candidates)}")
        lines.append(f"제외된 상품 수: {len(self.skipped_items)}")
        lines.append(f"생성된 발주 수: {len(created_orders)}")

        if created_orders:
            lines.append("\n[생성된 발주]")
            for order in created_orders:
                lines.append(
                    f"- {order['발주번호']} | {order['지점명']} | {order['업체코드']} | "
                    f"항목 {order['발주항목수']}개 | 수량 {order['총주문수량']}"
                )

        if self.skipped_items:
            lines.append("\n[제외된 상품]")
            for item in self.skipped_items:
                lines.append(
                    f"- {item['지점명']} | {item['상품코드']} | {item['상품명']} | "
                    f"{item['제외사유']}"
                )

        lines.append("")

        with log_file.open("a", encoding="utf-8") as f:
            f.write("\n".join(lines))

        print(f"\n로그 저장 완료: {log_file}")


# 표 형태로 결과 출력
def print_dict_rows(rows, max_width=22):
    if not rows:
        print("조회 결과가 없습니다.")
        return

    headers = list(rows[0].keys())
    str_rows = []

    for row in rows:
        str_rows.append([format_cell(row.get(header), max_width) for header in headers])

    widths = []
    for idx, header in enumerate(headers):
        width = max(len(header), *(len(row[idx]) for row in str_rows))
        widths.append(min(width, max_width))

    header_line = " | ".join(headers[i].ljust(widths[i]) for i in range(len(headers)))
    print("\n" + header_line)
    print("-" * len(header_line))

    for row in str_rows:
        print(" | ".join(row[i].ljust(widths[i]) for i in range(len(headers))))

    print(f"\n총 {len(rows)}행")


# 너무 긴 값은 짧게 줄여서 보여주기
def format_cell(value, max_width):
    if value is None:
        text = "NULL"
    else:
        text = str(value)

    text = text.replace("\n", " ")

    if len(text) > max_width:
        return text[: max_width - 3] + "..."

    return text


# DB 파일 자동으로 찾기
def discover_db_paths():
    paths = []

    cwd = Path.cwd()
    home = Path.home()

    # 폴더 위치가 바뀌어도 찾을 수 있도록 여러 위치를 후보로 둠
    project_roots = [
        cwd,
        cwd.parent,
        home / "paris-baguette-system-db",
        home / "Desktop" / "paris-baguette-system-db",
    ]

    # 같은 위치가 여러 번 들어가지 않게 정리
    unique_roots = []
    for root in project_roots:
        if root not in unique_roots:
            unique_roots.append(root)

    for root in unique_roots:
        search_dirs = [
            root / "schema",
            root / "data",
            root,
        ]

        for base in search_dirs:
            for name in DEFAULT_DB_NAMES:
                candidate = base / name
                if candidate.exists() and candidate not in paths:
                    paths.append(candidate)

    return paths


# 사용할 DB 파일 선택
def select_db_path():
    found = discover_db_paths()

    if found:
        # schema 폴더의 파리바게트.db를 가장 먼저 사용
        for path in found:
            if path.name == "파리바게트.db" and path.parent.name == "schema":
                print(f"DB 파일 자동 선택: {path}")
                return str(path)

        print(f"DB 파일 자동 선택: {found[0]}")
        return str(found[0])

    raise FileNotFoundError(
        "DB 파일을 찾을 수 없습니다. schema/파리바게트.db 위치를 확인해주세요."
    )


# 실행 옵션 받기
def parse_args():
    parser = argparse.ArgumentParser(
        description="파리바게트 자동 재주문 발주 프로그램"
    )
    parser.add_argument(
        "--db",
        help="사용할 DB 파일 경로",
    )
    parser.add_argument(
        "--execute",
        action="store_true",
        help="미리보기 없이 실제 발주까지 생성",
    )
    parser.add_argument(
        "--no-log",
        action="store_true",
        help="로그 파일 저장 안 함",
    )

    # 주피터 노트북에서 실행할 때 생기는 옵션은 무시하기 위해 parse_known_args 사용
    args, _ = parser.parse_known_args()
    return args


# 프로그램 시작
def main():
    args = parse_args()

    print("자동 재주문 발주 프로그램을 시작합니다.")
    print("기본 실행은 미리보기입니다. y를 입력하기 전까지 DB는 바뀌지 않습니다.")

    db_path = args.db if args.db else select_db_path()

    program = AutoReorderProgram(db_path)

    try:
        # 1. DB 연결
        program.connect()

        # 2. 재고 부족 상품 찾기
        candidates = program.scan_low_stock_items()

        # 3. 발주 예정 목록 보여주기
        program.print_summary(candidates)

        created_orders = []
        mode = "미리보기"

        if not candidates:
            print("\n새로 발주할 상품이 없어 종료합니다.")
        else:
            should_execute = args.execute

            # 4. 실제로 발주를 만들지 한 번 더 확인
            if not args.execute:
                print("\n현재는 미리보기 상태입니다.")
                print("실제로 발주를 만들려면 아래 질문에 y를 입력하세요.")
                confirm = input("발주를 실제로 생성할까요? (y/N) > ").strip().lower()
                should_execute = confirm == "y"

            # 5. 사용자가 동의하면 DB에 INSERT
            if should_execute:
                created_orders = program.execute_order_plan()
                program.print_created_orders(created_orders)
                mode = "실행"
            else:
                print("\n발주 생성을 취소했습니다. DB는 변경되지 않았습니다.")

        # 6. 로그 저장
        if not args.no_log:
            program.write_log(mode, candidates, created_orders)

    except sqlite3.Error as e:
        print(f"SQLite 오류가 발생했습니다: {e}")
    except Exception as e:
        print(f"오류가 발생했습니다: {e}")
    finally:
        program.close()


if __name__ == "__main__":
    main()
